In [ ]:
%matplotlib inline

import subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from datetime import datetime
plt.style.use('ggplot')

from pylab import rcParams
rcParams['figure.figsize'] = (10, 6)

import matplotlib
font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 17}
matplotlib.rc('font', **font)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# functions to read data and meta data
def read_data_given_id(path,ids,progress=True,last_offset=0):
    '''read data given a list of ids and CSV paths'''
    start = datetime.now()
    n = len(ids)
    if n == 0:
        return {}
    else:
        data = {}
        for (i,ist_id) in enumerate(ids, start=1):
            if progress and np.mod(i,np.ceil(n/10))==0:
                print('%d/%d (%2.0f%s) have been read...\t time consumed: %ds'\
                      %(i,n,i/n*100,'%',(datetime.now()-start).seconds))
            if last_offset==0:
                data[ist_id] = np.genfromtxt(path+str(ist_id)+'.csv',delimiter=',',\
                                         names='current,voltage',dtype=(float,float))
            else:
                p=subprocess.Popen(['tail','-'+str(int(last_offset)),path+str(ist_id)+'.csv'],\
                                   stdout=subprocess.PIPE)
                data[ist_id] = np.genfromtxt(p.stdout,delimiter=',',names='current,voltage',dtype=(float,float))
        print('%d/%d (%2.0f%s) have been read(Done!) \t time consumed: %ds'\
            %(n,n,100,'%',(datetime.now()-start).seconds)) 
        return data


def clean_meta(ist):
    '''remove None elements in Meta Data ''' 
    clean_ist = ist.copy()
    for k,v in ist.items():
        if len(v) == 0:
            del clean_ist[k]
    return clean_ist
                
def parse_meta(meta):
    '''parse meta data for easy access'''
    M = {}
    for m in meta:
        for app in m:
            M[int(app['id'])] = clean_meta(app['meta'])
    return M


In [ ]:
# specify path to read data and meta
# Please make sure you downloaded latest dataset from plaidplug.com.
Data_path = 'data/'
csv_path = Data_path + '2014/2014/';

import json

# read meta
with open(Data_path + 'meta_2014.json') as data_file:    
    meta1 = json.load(data_file)

    
Meta = parse_meta([meta1])
meta1 = parse_meta([meta1])

In [ ]:
# read data
# applinace types of all instances and map them to behaviour labels
# from behaviour_labels import BEHAVIOR_LABEL_MAP2

BEHAVIOR_LABEL_MAP2 = {
    "Fan": "SMALL_MOTOR_ELECTRONICS",
    "Vacuum": "SMALL_MOTOR_ELECTRONICS",
    "Washing Machine": "LAUNDRY_APPLIANCES",
    "Hairdryer": "THERMAL_APPLIANCES",
    "Fridge": "HVAC_REFRIGERATION",
    "Air Conditioner": "HVAC_REFRIGERATION",
    "Heater": "THERMAL_APPLIANCES",
    "Incandescent Light Bulb": "LIGHTING_LOADS",
    "Microwave": "THERMAL_APPLIANCES",
    "Laptop": "SMALL_MOTOR_ELECTRONICS",
    "Compact Fluorescent Lamp": "LIGHTING_LOADS"
}

Types = [BEHAVIOR_LABEL_MAP2[x['appliance']['type']] for x in Meta.values()]
# unique appliance types
Unq_type = list(set(Types)) 
Unq_type.sort()
IDs_for_read_data = list(Meta.keys())
# households of appliances
# Locs = [x['header']['collection_time']+'_'+x['location'] for x in Meta.values()]
# # unique households
# Unq_loc = list(set(Locs))
# Unq_loc.sort()
# Origianl_Unq_type = Unq_type

In [ ]:
# read data
# estimated time cost:  ~ 1 mins
npts = 0
Data = read_data_given_id(csv_path,IDs_for_read_data,progress=True, last_offset=npts)

In [ ]:
print('Total number of instances:%d'%len(Data))

Extract labels for appliance type and locations(used for train/test split).

In [ ]:
type_Ids = {}
loc_Ids = {}
n = len(Data)
type_label = np.zeros(n,dtype='int')
# loc_label = np.zeros(n,dtype='int')
for (ii,t) in enumerate(Unq_type):
    type_Ids[t] = [i-1 for i,j in enumerate(Types,start=1) if j == t]
    type_label[type_Ids[t]] = ii+1
# for (ii,t) in enumerate(Unq_loc):
#     loc_Ids[t] = [i-1 for i,j in enumerate(Locs,start=1) if j == t]
#     loc_label[loc_Ids[t]] = ii+1
print('number of different types: %d'% len(Unq_type))

In [ ]:
label2id = {key: value+1 for value, key in enumerate(Unq_type)}
id2label = {key+1: value for key, value in enumerate(Unq_type)}

### Save ID, label mapping

In [ ]:
# config = {
#     "id2label": id2label,
#     "label2id": label2id
# }

# with open("../models/model_config.json", "w") as f:
#     json.dump(config, f)

In [ ]:
type_Ids.keys()

In [ ]:
np.unique(type_label)

We transform high-frequency waveform datasets into low-frequency power envelopes that match the sensing capabilities of a smart plug. This prevents feature domain mismatch between training and deployment environments.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from window_processor import SlidingWindowProcessor
from training.plaid_adapter import waveform_to_power_series

def build_dataset(Data, labels, window_size=20, step_size=5):
    """
    Converts PLAID dataset dictionary into ML-ready feature matrix.
    """

    X = []
    y = []

    for instance_id in Data:

        voltage = Data[instance_id]['voltage']
        current = Data[instance_id]['current']
        label = labels[instance_id - 1]

        # convert waveform → smart plug signal
        power_series = waveform_to_power_series(voltage, current, target_hz=10)

        processor = SlidingWindowProcessor(window_size, step_size)

        for t, power in enumerate(power_series):
            features = processor.add_sample(t, power)
            if features:
                X.append(list(features.values()))
                y.append(label)

    return np.array(X), np.array(y)


In [ ]:
np.unique(type_label)

In [ ]:
X, y = build_dataset(Data, type_label)

In [ ]:
X

In [ ]:
X.shape

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
X2 = pca.fit_transform(X)

plt.figure(figsize=(8,6))
plt.scatter(X2[:,0], X2[:,1], c=y, s=5, cmap='tab20')
plt.title("Feature Space Projection (Before ML)")
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=42
)

model.fit(X, y)


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict

scores = cross_val_score(model, X, y, cv=5)
y_pred = cross_val_predict(model, X, y, cv=5)
print(scores.mean(), scores.std())


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


target_names = [id2label[i] for i in sorted(id2label.keys())]

cm = confusion_matrix(y, y_pred)
print(target_names)
print(cm)
ConfusionMatrixDisplay(cm, display_labels=target_names).plot(xticks_rotation=90)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

importance = model.feature_importances_
features = [
            "mean_power",
            "max_power",
            "min_power",
            "std_power",
            "range_power",
            "coef_var",
            "max_delta",
            "mean_delta",
            "spike_count",
            "slope",
            "oscillations"
        ]
names = features

pd.Series(importance, index=names).sort_values().plot.barh()
plt.title("Feature Importance")
plt.show()


## Since our contribution lies in sensing and system integration rather than classifier novelty, we validate the model through interpretability and behavior consistency instead of model competition.

In [ ]:
import joblib

joblib.dump(model, "../models/behaviour_classifier.pkl")

# Stat extraction for Anomaly Detection

In [ ]:
import json
import numpy as np
import os

feature_dim = X.shape[1]

anomaly_stats = {}

classes = np.unique(y)

for cls in classes:
    X_cls = X[y == cls]

    mean_vec = X_cls.mean(axis=0)
    std_vec = X_cls.std(axis=0)

    # Avoid divide-by-zero during inference
    std_vec[std_vec < 1e-6] = 1e-6

    anomaly_stats[cls] = {
        "mean": mean_vec.tolist(),
        "std": std_vec.tolist()
    }



In [ ]:
{int(i):j for i, j in anomaly_stats.items()}

### Save anomaly stats

In [ ]:
os.makedirs("../models", exist_ok=True)

with open("../models/anomaly_stats.json", "w") as f:
    json.dump({int(i):j for i, j in anomaly_stats.items()}, f, indent=2)

print("Exported anomaly_stats.json")